# 03b Run Modality Verification

Objective: verify deterministic Task 2 extractions with a source-grounded Task 3 self-verification prompt.

This diagnostic does not revise Task 2 outputs. It asks the same model whether its extracted requirement preserves, strengthens, weakens, or changes the source.


In [ ]:
from pathlib import Path
import os
import sys
import importlib

PROJECT_ROOT = Path.cwd()
if PROJECT_ROOT.name == "notebooks":
    PROJECT_ROOT = PROJECT_ROOT.parent
while not (PROJECT_ROOT / "AGENTS.md").exists() and PROJECT_ROOT.parent != PROJECT_ROOT:
    PROJECT_ROOT = PROJECT_ROOT.parent

sys.path.insert(0, str(PROJECT_ROOT / "scripts"))
import eval_utils as eu
eu = importlib.reload(eu)

CONFIG_PATH = PROJECT_ROOT / "config.json"
if not CONFIG_PATH.exists():
    CONFIG_PATH = PROJECT_ROOT / "config.example.json"
CONFIG = eu.load_config(CONFIG_PATH)
eu.ensure_project_dirs(PROJECT_ROOT)
BENCHMARK_VARIANT = os.getenv("BENCHMARK_VARIANT", "must").strip().lower()
VARIANT_SUFFIX = eu.variant_suffix(BENCHMARK_VARIANT)

PROJECT_ROOT, CONFIG_PATH, BENCHMARK_VARIANT


## Configure Verification Run


In [ ]:
HOST = os.getenv("HOST", CONFIG["llm"]["host"])
RUN_TASK3_VERIFICATION = os.getenv("RUN_TASK3_VERIFICATION", "true").lower() in {"1", "true", "yes"}
deterministic = CONFIG["llm"]["deterministic"]
stochastic = CONFIG["llm"]["stochastic"]
REQUEST_CONCURRENCY = eu.resolve_llm_concurrency(CONFIG)

benchmark_path = eu.variant_path(PROJECT_ROOT / "data/processed/benchmark_items.csv", BENCHMARK_VARIANT)
source_raw_path = eu.variant_path(PROJECT_ROOT / "data/processed/model_outputs_raw.jsonl", BENCHMARK_VARIANT)
task3_items_path = eu.variant_path(PROJECT_ROOT / "data/processed/task3_verification_items.csv", BENCHMARK_VARIANT)
output_path = eu.variant_path(PROJECT_ROOT / "data/processed/model_outputs_raw_task3_verification.jsonl", BENCHMARK_VARIANT)

benchmark = eu.read_csv_rows(benchmark_path)
all_source_rows = eu.read_jsonl(source_raw_path)
requested_source_run_id = os.getenv("TASK3_SOURCE_RUN_ID") or os.getenv("RUN_ID")
run_prefix = "full" if BENCHMARK_VARIANT == "must" else f"full-{BENCHMARK_VARIANT}"

if requested_source_run_id:
    source_run_id, source_rows = eu.select_run_rows(all_source_rows, run_id=requested_source_run_id, prefix=run_prefix)
else:
    progress = eu.run_progress_summary(
        benchmark,
        all_source_rows,
        expected_stochastic_samples=int(stochastic["samples"]),
    )
    complete_run_ids = eu.complete_run_ids_from_progress(progress, prefix=run_prefix)
    if not complete_run_ids:
        available = sorted({row.get("run_id", "") for row in all_source_rows if str(row.get("run_id", "")).startswith(run_prefix)})
        raise ValueError(
            "No complete full run found for Task 3 verification. "
            f"Set TASK3_SOURCE_RUN_ID explicitly or finish a full run. Available run_ids: {available[-10:]}"
        )
    source_run_id, source_rows = eu.select_run_rows(all_source_rows, run_id=complete_run_ids[-1], prefix=run_prefix)

run_id = eu.new_run_id("task3" if BENCHMARK_VARIANT == "must" else f"task3-{BENCHMARK_VARIANT}")
print({
    "HOST": HOST,
    "RUN_TASK3_VERIFICATION": RUN_TASK3_VERIFICATION,
    "REQUEST_CONCURRENCY": REQUEST_CONCURRENCY,
    "source_run_id": source_run_id,
    "task3_run_id": run_id,
    "BENCHMARK_VARIANT": BENCHMARK_VARIANT,
})
print(f"Benchmark path: {benchmark_path}")
print(f"Task 1/2 raw path: {source_raw_path}")
print(f"Task 3 output path: {output_path}")


## Build Task 3 Verification Items


In [ ]:
task3_items = eu.build_task3_verification_items(benchmark, source_rows)
eu.write_csv_rows(task3_items_path, task3_items, fieldnames=eu.TASK3_VERIFICATION_FIELDS)
print(f"Wrote Task 3 verification items: {task3_items_path}")
print(f"Task 3 items: {len(task3_items)}")
if not task3_items:
    raise ValueError("No Task 3 items were built. Check that the selected run has valid deterministic Task 2 rows.")
print(eu.markdown_table(task3_items[:8], ["item_id", "source_modality", "task2_modality", "task3_gold_relation"]))


## Run Task 3 Verification


In [ ]:
task3_template = eu.load_prompt(PROJECT_ROOT / "prompts/modality_verification.txt")

def task3_prompt_for(item):
    return eu.render_prompt(
        task3_template,
        source_statement=item["source_statement"],
        extracted_requirement=item["task2_requirement"],
        extracted_modality=item["task2_modality"],
    )

def task3_request_job(item, sample_kind, sample_index, temperature, top_p, request_index):
    return {
        "request_index": request_index,
        "run_id": run_id,
        "model": item["task2_model"],
        "host": HOST,
        "task": "task3",
        "item": item,
        "sample_index": sample_index,
        "sample_kind": sample_kind,
        "temperature": temperature,
        "top_p": top_p,
        "prompt_version": f"{CONFIG['project']['prompt_version']}:task3",
        "prompt": task3_prompt_for(item),
        "max_tokens": int(CONFIG["llm"]["max_tokens"]),
        "timeout_s": int(CONFIG["llm"]["timeout_s"]),
        "api_key_env": CONFIG["llm"]["api_key_env"],
    }

jobs = []
for item in task3_items:
    jobs.append(task3_request_job(
        item=item,
        sample_kind="deterministic",
        sample_index=0,
        temperature=float(deterministic["temperature"]),
        top_p=float(deterministic["top_p"]),
        request_index=len(jobs),
    ))
    for sample_index in range(int(stochastic["samples"])):
        jobs.append(task3_request_job(
            item=item,
            sample_kind="stochastic",
            sample_index=sample_index,
            temperature=float(stochastic["temperature"]),
            top_p=float(stochastic["top_p"]),
            request_index=len(jobs),
        ))

planned_calls = len(task3_items) * (1 + int(stochastic["samples"]))
assert len(jobs) == planned_calls
records = []

if RUN_TASK3_VERIFICATION:
    print(f"Dispatching {len(jobs)} Task 3 calls with concurrency={REQUEST_CONCURRENCY}")
    for record in eu.run_completion_jobs(jobs, max_workers=REQUEST_CONCURRENCY):
        eu.append_jsonl(output_path, record)
        records.append(record)
        if len(records) % 50 == 0 or len(records) == len(jobs):
            print(f"Completed {len(records)}/{len(jobs)} Task 3 calls")
    print(f"Wrote {len(records)} Task 3 records to {output_path}")
else:
    print("Task 3 verification not run. Set RUN_TASK3_VERIFICATION=true to execute it.")


## Verification Summary


In [ ]:
task3_rows = [row for row in eu.read_jsonl(output_path) if row.get("run_id") == run_id] if RUN_TASK3_VERIFICATION else []
if task3_rows:
    status_counts = {}
    for row in task3_rows:
        status_counts[row["parse_status"]] = status_counts.get(row["parse_status"], 0) + 1
    print(status_counts)
    print(f"Parse success rate: {status_counts.get('ok', 0) / len(task3_rows):.3f}")
    task3_scores = eu.build_task3_scores(task3_items, task3_rows)
    summary = eu.metric_summary_by_model_task_method(task3_scores)
    fields = [
        "model",
        "task",
        "uq_method",
        "n",
        "accuracy",
        "f1_or_macro_f1",
        "strengthening_recall",
        "false_preserve_rate",
        "evidence_phrase_source_rate",
        "brier",
        "ece",
        "error_detection_auroc",
        "parse_failure_rate",
    ]
    print(eu.markdown_table(summary, fields))
else:
    print("No Task 3 rows for this run_id yet.")
